<a href="https://colab.research.google.com/github/chaitanya731-spec/automated-question-answering/blob/main/Automatic_MCQ_Generation_DL_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Automatic Multiple-Choice Question Generation using Transformer & LLMs
### Deep Learning Course Project

This notebook implements an **end-to-end Automatic Multiple-Choice Question Generation (MCQG) pipeline**, built on the taxonomy described in:

> H. W. Awalurahman, R. F. Aji, and I. Budi, *"Transformer and Large Language Models for Automatic Multiple-Choice Question Generation: A Systematic Literature Review,"* IEEE Access, vol. 13, 2025.

The paper identifies two major strategies used in the literature:

1. **Transformer-based approach (fine-tuning)** — a Transformer model (e.g., T5) is fine-tuned on a QA dataset (e.g., SQuAD) for **Question Generation (QG)**, often paired with a **lexical / Transformer-based Distractor Generation (DG)** model.
2. **LLM-based approach (prompting)** — a large pretrained LLM (GPT, Gemini, Llama, etc.) generates the full MCQ (question + answer + distractors) using zero-shot / few-shot / chain-of-thought / RAG prompting, with **no fine-tuning**.

This project implements **both strategies** so they can be compared, exactly as RQ1 of the paper investigates:

| Component | Method used here | Maps to paper's taxonomy |
|---|---|---|
| Question Generation | Fine-tuned **T5** (highlight strategy: `<hl> answer <hl>` in context) | Transformer-based, fine-tuning, "highlight" input format (Table 7 / 8) |
| Distractor Generation | **BERT masked-language-model** substitution + semantic-similarity filtering | Transformer-based DG (lexical/embedding-driven distractors) |
| End-to-end baseline | **Zero-shot prompting** with an instruction-tuned LLM (optional, via API) | LLM-based approach, zero-shot prompting (Table 2/3) |
| Evaluation | **BLEU / ROUGE** (automatic) + a manual-quality rubric (relevance, plausibility, fluency) | RQ2 — automatic vs. manual evaluation (Table 4/5/6) |

---
**How to run:** `Runtime > Run all`. GPU runtime is recommended (`Runtime > Change runtime type > T4 GPU`).


## 1. Setup

In [13]:
!pip install -q transformers datasets evaluate accelerate sentencepiece rouge_score sacrebleu sentence-transformers


In [14]:
import os
import random
import numpy as np
import torch
import torch.nn.functional as F

from datasets import load_dataset
from transformers import (
    T5TokenizerFast, T5ForConditionalGeneration,
    Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq,
    AutoTokenizer, AutoModelForMaskedLM,
)
from sentence_transformers import SentenceTransformer, util
import evaluate

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)


Using device: cuda


## 2. Dataset

We use **SQuAD** (Stanford Question Answering Dataset), which the paper reports as the most frequently used dataset for Transformer-based fine-tuning studies (see Section IV-A: *"A notable trend is the frequent use of the SQuAD dataset..."*).

For a course project we take a manageable subset so the whole notebook trains in a reasonable time on a single Colab GPU. Increase `TRAIN_SIZE` / `NUM_EPOCHS` for a stronger model if you have more compute time available.

In [15]:
TRAIN_SIZE = 4000   # increase for better quality if you have time/GPU
VAL_SIZE   = 500

raw = load_dataset("rajpurkar/squad")
train_raw = raw["train"].shuffle(seed=SEED).select(range(TRAIN_SIZE))
val_raw   = raw["validation"].shuffle(seed=SEED).select(range(VAL_SIZE))

print(train_raw[0])


{'id': '573173d8497a881900248f0c', 'title': 'Egypt', 'context': 'The Pew Forum on Religion & Public Life ranks Egypt as the fifth worst country in the world for religious freedom. The United States Commission on International Religious Freedom, a bipartisan independent agency of the US government, has placed Egypt on its watch list of countries that require close monitoring due to the nature and extent of violations of religious freedom engaged in or tolerated by the government. According to a 2010 Pew Global Attitudes survey, 84% of Egyptians polled supported the death penalty for those who leave Islam; 77% supported whippings and cutting off of hands for theft and robbery; and 82% support stoning a person who commits adultery.', 'question': 'What percentage of Egyptians polled support death penalty for those leaving Islam?', 'answers': {'text': ['84%'], 'answer_start': [468]}}


## 3. Question-Generation Preprocessing — "Highlight" Strategy

The paper (Table 7/8) describes two common fine-tuning input formats:

- **Highlight**: tag the answer span directly inside the context, e.g. `generate question: context <hl> answer <hl> context`
- **Prepend**: put the answer before the context, e.g. `answer: <answer> context: <context>`

We implement the **highlight** strategy (used by Goyal et al. [17] in the paper), since it explicitly points the model to the answer span rather than relying purely on string matching.

In [16]:
HL_TOKEN = "<hl>"

def build_qg_input(context, answer_text, answer_start):
    """Insert <hl> ... <hl> around the answer span inside the context."""
    answer_end = answer_start + len(answer_text)
    highlighted = (
        context[:answer_start] + f"{HL_TOKEN} " + context[answer_start:answer_end] +
        f" {HL_TOKEN}" + context[answer_end:]
    )
    return f"generate question: {highlighted}"

def preprocess_example(example):
    answer_text = example["answers"]["text"][0]
    answer_start = example["answers"]["answer_start"][0]
    src = build_qg_input(example["context"], answer_text, answer_start)
    tgt = example["question"]
    return {"input_text": src, "target_text": tgt, "answer_text": answer_text}

train_ds = train_raw.map(preprocess_example, remove_columns=train_raw.column_names)
val_ds   = val_raw.map(preprocess_example, remove_columns=val_raw.column_names)

print(train_ds[0]["input_text"][:300])
print("--->", train_ds[0]["target_text"])


generate question: The Pew Forum on Religion & Public Life ranks Egypt as the fifth worst country in the world for religious freedom. The United States Commission on International Religious Freedom, a bipartisan independent agency of the US government, has placed Egypt on its watch list of countries
---> What percentage of Egyptians polled support death penalty for those leaving Islam?


## 4. Tokenization

In [17]:
MODEL_NAME = "t5-small"   # try "t5-base" if you have more GPU memory/time
MAX_INPUT_LEN = 384
MAX_TARGET_LEN = 64

tokenizer = T5TokenizerFast.from_pretrained(MODEL_NAME)
tokenizer.add_special_tokens({"additional_special_tokens": [HL_TOKEN]})

def tokenize_fn(batch):
    model_inputs = tokenizer(
        batch["input_text"], max_length=MAX_INPUT_LEN, truncation=True, padding="max_length"
    )
    labels = tokenizer(
        text_target=batch["target_text"], max_length=MAX_TARGET_LEN, truncation=True, padding="max_length"
    )
    labels["input_ids"] = [
        [(t if t != tokenizer.pad_token_id else -100) for t in seq] for seq in labels["input_ids"]
    ]
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_tok = train_ds.map(tokenize_fn, batched=True, remove_columns=train_ds.column_names)
val_tok   = val_ds.map(tokenize_fn, batched=True, remove_columns=["input_text", "target_text"])


## 5. Fine-tune T5 for Question Generation

This is the **fine-tuning branch** of the paper's taxonomy (Fig. 3): `Transformer -> Input-Output Structure -> Highlight`.

In [18]:
import inspect

model = T5ForConditionalGeneration.from_pretrained(MODEL_NAME)
model.resize_token_embeddings(len(tokenizer))
model.to(DEVICE)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

# transformers renamed `evaluation_strategy` -> `eval_strategy` across versions; support both.
training_args_kwargs = dict(
    output_dir="./t5-qg-checkpoints",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=3e-4,
    num_train_epochs=3,
    weight_decay=0.01,
    save_strategy="epoch",
    save_total_limit=1,
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),
    logging_steps=50,
    report_to="none",
)

args_params = inspect.signature(Seq2SeqTrainingArguments.__init__).parameters
if "eval_strategy" in args_params:
    training_args_kwargs["eval_strategy"] = "epoch"
else:
    training_args_kwargs["evaluation_strategy"] = "epoch"

training_args = Seq2SeqTrainingArguments(**training_args_kwargs)

# transformers renamed the `tokenizer` kwarg to `processing_class` in newer releases;
# detect which one this installed version expects.
trainer_kwargs = dict(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    data_collator=data_collator,
)

trainer_params = inspect.signature(Seq2SeqTrainer.__init__).parameters
if "processing_class" in trainer_params:
    trainer_kwargs["processing_class"] = tokenizer
else:
    trainer_kwargs["tokenizer"] = tokenizer

trainer = Seq2SeqTrainer(**trainer_kwargs)

trainer.train()

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss
1,2.051502,1.820854
2,1.765022,1.781727
3,1.571151,1.783137


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1500, training_loss=1.8671583709716797, metrics={'train_runtime': 306.7289, 'train_samples_per_second': 39.122, 'train_steps_per_second': 4.89, 'total_flos': 1218076213248000.0, 'train_loss': 1.8671583709716797, 'epoch': 3.0})

In [19]:
SAVE_DIR = "./t5-qg-model"
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print("Saved fine-tuned QG model to", SAVE_DIR)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved fine-tuned QG model to ./t5-qg-model


## 6. Inference — Question Generation

In [20]:
qg_tokenizer = T5TokenizerFast.from_pretrained(SAVE_DIR)
qg_model = T5ForConditionalGeneration.from_pretrained(SAVE_DIR).to(DEVICE)
qg_model.eval()

def generate_question(context, answer_text, answer_start=None, num_beams=4, max_length=64):
    if answer_start is None:
        answer_start = context.find(answer_text)
        if answer_start == -1:
            raise ValueError("answer_text not found in context; please pass answer_start explicitly")
    src = build_qg_input(context, answer_text, answer_start)
    inputs = qg_tokenizer(src, return_tensors="pt", truncation=True, max_length=MAX_INPUT_LEN).to(DEVICE)
    with torch.no_grad():
        out = qg_model.generate(
            **inputs, max_length=max_length, num_beams=num_beams, early_stopping=True
        )
    return qg_tokenizer.decode(out[0], skip_special_tokens=True)


sample_context = (
    "The Transformer architecture was introduced by Vaswani et al. in 2017 in the paper "
    "'Attention Is All You Need'. It relies entirely on a self-attention mechanism, removing "
    "the need for recurrence and convolutions used in earlier sequence models."
)
sample_answer = "Vaswani et al."

print("Generated question:", generate_question(sample_context, sample_answer))


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Generated question: Who introduced Transformer architecture in 2017?


## 7. Distractor Generation (Transformer-based, BERT Masked-LM)

The paper's taxonomy pairs a QG Transformer with a **lexical/embedding-based distractor generator** (e.g., Sense2Vec, Word2Vec, WordNet). Since those pretrained lexical resources are large external downloads, we implement an equivalent, self-contained strategy using **BERT's masked-language-model head**:

1. Mask the answer span inside the context.
2. Ask BERT to predict the most likely tokens/words for that position.
3. Filter out candidates that are (a) the correct answer itself, (b) not plausible English words, or (c) too semantically *close* to the correct answer (which would make them arguably also-correct) — using **Sentence-BERT cosine similarity**, keeping a "plausible but wrong" middle band, mirroring the paper's discussion of *plausibility* as a manual evaluation criterion (Table 6).

In [21]:
MLM_NAME = "bert-base-uncased"
mlm_tokenizer = AutoTokenizer.from_pretrained(MLM_NAME)
mlm_model = AutoModelForMaskedLM.from_pretrained(MLM_NAME).to(DEVICE)
mlm_model.eval()

sbert = SentenceTransformer("all-MiniLM-L6-v2", device=DEVICE)

def generate_distractors(context, answer_text, top_k=25, num_distractors=3,
                          sim_low=0.15, sim_high=0.75):
    """Generate plausible-but-incorrect distractors for `answer_text`."""
    idx = context.find(answer_text)
    if idx == -1:
        raise ValueError("answer_text not found in context")

    masked_context = context[:idx] + mlm_tokenizer.mask_token + context[idx + len(answer_text):]
    inputs = mlm_tokenizer(masked_context, return_tensors="pt", truncation=True, max_length=384).to(DEVICE)
    mask_positions = (inputs["input_ids"][0] == mlm_tokenizer.mask_token_id).nonzero(as_tuple=True)[0]

    if len(mask_positions) == 0:
        return []

    with torch.no_grad():
        logits = mlm_model(**inputs).logits
    mask_logits = logits[0, mask_positions[0]]
    top_ids = torch.topk(mask_logits, top_k).indices.tolist()
    candidates = [mlm_tokenizer.decode([tid]).strip() for tid in top_ids]

    # Basic cleanup: alphabetic words only, not the answer itself, deduplicated
    candidates = [c for c in candidates if c.isalpha() and c.lower() != answer_text.lower()]
    candidates = list(dict.fromkeys(candidates))

    if not candidates:
        return []

    # Semantic-similarity filtering: keep candidates that are "related but wrong"
    answer_emb = sbert.encode(answer_text, convert_to_tensor=True)
    cand_embs = sbert.encode(candidates, convert_to_tensor=True)
    sims = util.cos_sim(answer_emb, cand_embs)[0].tolist()

    scored = [(c, s) for c, s in zip(candidates, sims) if sim_low <= s <= sim_high]
    scored.sort(key=lambda x: -x[1])  # most plausible (closest but still wrong) first

    distractors = [c for c, _ in scored[:num_distractors]]

    # Fallback: if similarity filtering removed too many, backfill from remaining candidates
    if len(distractors) < num_distractors:
        backfill = [c for c in candidates if c not in distractors]
        distractors += backfill[: num_distractors - len(distractors)]

    return distractors[:num_distractors]


print("Distractors:", generate_distractors(sample_context, sample_answer))


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Distractors: ['colleagues', 'researchers', 'lee']


## 8. End-to-End MCQ Pipeline

Combines the fine-tuned T5 QG model with the BERT-based DG module, matching the **"Step-Wise" fine-tuning approach** in Table 7 of the paper (question generated first, then distractors), as opposed to a single end-to-end LLM call.

In [22]:
def generate_mcq(context, answer_text, answer_start=None, num_distractors=3):
    question = generate_question(context, answer_text, answer_start)
    distractors = generate_distractors(context, answer_text, num_distractors=num_distractors)

    options = distractors + [answer_text]
    random.shuffle(options)
    correct_index = options.index(answer_text)

    return {
        "context": context,
        "question": question,
        "options": options,
        "correct_answer": answer_text,
        "correct_index": correct_index,
    }


def pretty_print_mcq(mcq):
    print("Q:", mcq["question"])
    letters = "ABCD"
    for i, opt in enumerate(mcq["options"]):
        marker = " (correct)" if i == mcq["correct_index"] else ""
        print(f"   {letters[i]}. {opt}{marker}")


demo_mcq = generate_mcq(sample_context, sample_answer)
pretty_print_mcq(demo_mcq)


Q: Who introduced Transformer architecture in 2017?
   A. lee
   B. researchers
   C. Vaswani et al. (correct)
   D. colleagues


In [23]:
# Try it on your own paragraph + answer
my_context = """Photosynthesis is the process by which green plants and some other organisms
use sunlight to synthesize foods from carbon dioxide and water. Photosynthesis in plants
generally involves the green pigment chlorophyll and generates oxygen as a byproduct."""

my_answer = "chlorophyll"

mcq = generate_mcq(my_context, my_answer)
pretty_print_mcq(mcq)


Q: What is the green pigment used in photosynthesis?
   A. chlorophyll (correct)
   B. plant
   C. algae
   D. plants


## 9. (Optional) LLM-based Zero-Shot Baseline

The paper reports that most recent studies (17 of 28 in 2024) have moved toward **LLM-based, zero-shot prompting** rather than fine-tuning, because it needs no training data and generates the full MCQ in a single call (Table 2, Fig. 3). This cell lets you compare that approach against your fine-tuned pipeline above.

> This cell requires your own API key and is **optional** — skip it if you don't have API access. It is included to let you reproduce the "prompting" branch of the paper's taxonomy for your project report's comparison section (RQ1).

In [24]:
# Uncomment and fill in your API key to try the zero-shot LLM baseline.
# This mirrors the zero-shot prompt style shown in Table 3 of the paper.

# import anthropic
# client = anthropic.Anthropic(api_key="YOUR_API_KEY")
#
# ZERO_SHOT_PROMPT = '''Write a multiple choice question using the following sentence and answer.
# Convert the sentence into a question that matches the answer. Provide exactly 3 plausible but
# incorrect distractor options, and return the result as JSON with keys:
# "question", "options" (list of 4, shuffled), "correct_answer".
#
# Sentence: {context}
# Answer: {answer}'''
#
# def llm_zero_shot_mcq(context, answer):
#     msg = client.messages.create(
#         model="claude-sonnet-4-6",
#         max_tokens=500,
#         messages=[{"role": "user", "content": ZERO_SHOT_PROMPT.format(context=context, answer=answer)}],
#     )
#     return msg.content[0].text
#
# print(llm_zero_shot_mcq(sample_context, sample_answer))


## 10. Evaluation (RQ2 of the paper)

The paper finds that studies use **automatic**, **manual**, or **mixed** evaluation (Table 4). We implement:

- **Automatic**: BLEU and ROUGE-L between generated questions and the human-written reference questions from SQuAD's validation set — the most-used automatic metrics per Table 5.
- **Semi-automated plausibility check**: average semantic similarity of distractors to the correct answer (should be "related but not equal" — neither too similar nor completely unrelated), echoing the *Plausibility* and *Distinctiveness* manual criteria in Table 6.

In [25]:
bleu = evaluate.load("sacrebleu")
rouge = evaluate.load("rouge")

EVAL_N = 100  # number of validation examples to evaluate on
eval_subset = val_ds.select(range(min(EVAL_N, len(val_ds))))

predictions, references = [], []
for ex in eval_subset:
    # re-derive context/answer/answer_start since we removed columns during preprocessing;
    # easiest is to re-pull from val_raw by index
    pass

# Re-run generation directly against val_raw (keeps original fields available)
predictions, references = [], []
for ex in val_raw.select(range(min(EVAL_N, len(val_raw)))):
    answer_text = ex["answers"]["text"][0]
    answer_start = ex["answers"]["answer_start"][0]
    try:
        pred_q = generate_question(ex["context"], answer_text, answer_start)
    except Exception:
        continue
    predictions.append(pred_q)
    references.append([ex["question"]])

bleu_score = bleu.compute(predictions=predictions, references=references)
rouge_score = rouge.compute(predictions=predictions, references=[r[0] for r in references])

print(f"Evaluated on {len(predictions)} examples")
print("BLEU:", round(bleu_score["score"], 2))
print("ROUGE-L:", round(rouge_score["rougeL"], 4))


Evaluated on 100 examples
BLEU: 15.37
ROUGE-L: 0.4245


In [26]:
# Semi-automated distractor plausibility check
sims_all = []
for ex in val_raw.select(range(30)):
    answer_text = ex["answers"]["text"][0]
    try:
        distractors = generate_distractors(ex["context"], answer_text, num_distractors=3)
    except Exception:
        continue
    if not distractors:
        continue
    answer_emb = sbert.encode(answer_text, convert_to_tensor=True)
    dist_embs = sbert.encode(distractors, convert_to_tensor=True)
    sims = util.cos_sim(answer_emb, dist_embs)[0].tolist()
    sims_all.extend(sims)

if sims_all:
    print(f"Distractor-to-answer similarity over {len(sims_all)} distractors:")
    print("  mean:", round(np.mean(sims_all), 3), " std:", round(np.std(sims_all), 3))
    print("  (target band ~0.15-0.75: related but not synonymous with the correct answer)")


Distractor-to-answer similarity over 86 distractors:
  mean: 0.439  std: 0.16
  (target band ~0.15-0.75: related but not synonymous with the correct answer)


## 11. Discussion & Report Notes

Use these observations (mirrored from the paper's Discussion / Limitations sections) when writing your project report:

- **Fine-tuning vs. prompting trade-off (Section V-A):** fine-tuning T5 required a training dataset (SQuAD) and GPU time, but produces a small, fast, self-hosted model. An LLM zero-shot baseline needs no training data but depends on an external API and larger compute per call.
- **Distractor quality (Section V-A):** the paper notes that lexical methods (Sense2Vec/Word2Vec/WordNet) tend to produce **word/phrase-level** distractors, while Transformer-Transformer or LLM approaches can produce **sentence-level** distractors better suited to reading-comprehension-style MCQs. Our BERT-MLM approach is a word/phrase-level distractor generator, similar in spirit to the lexical family.
- **Evaluation limitation (Section V-B, RQ2):** automatic metrics like BLEU/ROUGE only measure *overlap* with a reference question, not actual pedagogical quality (clarity, difficulty, distractor plausibility). The paper stresses that most primary studies (12/28) therefore still rely on **manual/expert evaluation** — consider adding a small human-rating step (e.g., 3–5 raters scoring 10 generated MCQs on relevance/fluency/plausibility, as in Table 6) to strengthen your report.
- **Future work suggested by the paper** you could extend this project with: multilingual MCQ generation, chain-of-thought or RAG-based distractor generation, or Bloom's-taxonomy-conditioned question generation (Section V-D).
